<a href="https://colab.research.google.com/github/CloudConsultant2/awesome-json/blob/master/mpepllm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf sentence-transformers google-genai


In [3]:
from pypdf import PdfReader
# PDF ke har page ka text ek string me jodo
reader = PdfReader("/mpep-2100.pdf")   # <- apna exact filename
text = ""
for page in reader.pages:
    text += (page.extract_text() or "") + "\n"
print("Characters:", len(text))

Characters: 2443062


In [4]:
# Bada text chhote tukdon me — overlap taaki context na toote
def chunk_text(text, size=1000, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start+size])
        start += size - overlap
    return chunks
chunks = chunk_text(text)
print("Chunks:", len(chunks))

Chunks: 2875


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
# Local model — Colab me chalta, free, unlimited
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
embeddings = model.encode(chunks, show_progress_bar=True, normalize_embeddings=True)
print("Shape:", embeddings.shape)   # (2875, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

Shape: (2875, 384)


In [7]:
# Sawaal embed → sabse milte chunks (cosine)
def search(query, top_k=5):
    q = model.encode([query], normalize_embeddings=True)[0]
    sims = embeddings @ q
    top = sims.argsort()[-top_k:][::-1]
    return [chunks[i] for i in top]

In [9]:
from google import genai
from google.colab import userdata
client = genai.Client(api_key=userdata.get("GEMINI_KEY"))

# Mile chunks ko context bana ke jawab — MPEP se, memory se nahi
def ask(query):
    context = "\n\n".join(search(query))
    prompt = f"""Answer using ONLY the MPEP context below. Cite the relevant section number.
Context:
{context}

Question: {query}"""
    return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text

print(ask("What is the duty of disclosure under MPEP?"))

The provided MPEP context does not define or discuss the "duty of disclosure." The term "disclosure" is defined as a generic expression encompassing documents and activities enumerated in AIA 35 U.S.C. 102(a) (e.g., patented, described in a printed publication, in public use, on sale, or otherwise available to the public). This definition is found in MPEP § 2152.04. Other sections discuss the application of specific prior art provisions or the requirements for showing prior public disclosures, but not a general "duty of disclosure."


In [10]:
print(ask("What are the four categories of statutory subject matter under 35 U.S.C. 101?"))

The four categories of statutory subject matter under 35 U.S.C. 101 are processes, machines, manufactures, and compositions of matter. (MPEP § 2106.03, I. and MPEP § 2106, I.)


In [11]:
import pickle
with open("mpep_data.pkl", "wb") as f:
    pickle.dump({"chunks": chunks, "embeddings": embeddings}, f)

from google.colab import files
files.download("mpep_data.pkl")   # apne PC pe download ho jaayega

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>